In [4]:
script_name = "021_plot_2023_TW_Estimation"

# Project Setup

In [5]:
from pathlib import Path

# Assume this script is located in 02_code/
project_dir = Path("..").resolve()

# Public data directory
data_dir = project_dir / "01_data"

# Output base directory
output_dir = project_dir / "outputs"
output_dir.mkdir(exist_ok=True)


# Script-specific output directories
midstream_dir = output_dir / script_name / "midstream"
results_dir   = output_dir / script_name / "results"

# Create directories
midstream_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

# Library Import

In [6]:
import os
import glob
import numpy as np
import pandas as pd
import datetime as dt

import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

# Data Loading and Preprocessing

In [7]:
year_now = "2023"

# Specify when to use the trained model set
model_timestamp_config_df = pd.read_csv(data_dir / "0_model_timestamp_config.csv", index_col=0, dtype=str).apply(lambda col: col.str.strip('"'))

CV_modeltime_now = model_timestamp_config_df.loc["RandomCV", year_now]
blo_CV_modeltime_now = model_timestamp_config_df.loc["BlockwiseCV", year_now]

In [8]:
CV_modeltime_now

nan

# Main workflow

In [9]:
base_dir =  output_dir / r"021_2023_TW_Estimation\results" 
HarvTime_now = 3          
img_type = "Full"

## Model Parameters

In [10]:
# UAV flight date 
flight_date_li = ['202304171517', '202305031115', '202305181131', '202305301156', '202306121122', '202306271027']
mul_flight_date_li = ['202305031115', '202305181131', '202305301127', '202306121122', '202306271027']
latest_flight_date_li = ["2023" + s for s in ["0503","0518", "0530", "0612", "0627", "0718"]] 

# Calculate Days after Sowing
reference_date = pd.to_datetime('20230330', format='%Y%m%d')
samp_date_dic ={"1":"20230519", "2":"20230601", "3":"20230615", "4":"20230629", "5":"20230719"}
samp_DAS_dic = {str(s):(dt.datetime.strptime(samp_date_dic[str(s)],'%Y%m%d') - reference_date).days for s in range(1,6)}

# Feature names
RGB_Pheno_names = ["PH_90", "CC_rate", "PV_ave", "ExG_VI", "VARI_VI", "GRVI_VI"]
Mul_VI_names = ["NDVI", "GNDVI", "NDRE"]
Mul_Pheno_names = [s + "_median" for s in Mul_VI_names]
UAV_pheno_names = RGB_Pheno_names + Mul_Pheno_names

## Feature Selection Visualization

In [11]:
idxs = list(range(1+HarvTime_now))   
input_comb_date_now = [latest_flight_date_li[i] for i in idxs]

RGB_input_comb_now = [s for s in flight_date_li if s[:8] in input_comb_date_now]
RGB_input_UAV_cols = [
    f"{p}_{d}"
    for d in RGB_input_comb_now
    for p in RGB_Pheno_names
]

mul_input_comb_now = [s for s in mul_flight_date_li if s[:8] in input_comb_date_now]
mul_input_UAV_cols = [
    f"{p}_{d}"
    for d in mul_input_comb_now
    for p in Mul_Pheno_names
]

if img_type == "RGB":
    input_UAV_cols = RGB_input_UAV_cols
elif img_type == "Full":
    input_UAV_cols = RGB_input_UAV_cols + mul_input_UAV_cols

In [12]:
model_dir = os.path.join(
    base_dir,
    "CV10_" + "harv" + str(HarvTime_now),
    f"K10foldCVwithPipe5_{CV_modeltime_now}"
)

model_paths_all = glob.glob(
    os.path.join(model_dir, "*.pickele")
)

model_paths = [
    p for p in model_paths_all
    if ("SampRate-100" in os.path.basename(p)) 
    and ("SampMetho-random" in os.path.basename(p))
    and ("proto" not in os.path.basename(p))
]
print(len(model_paths), "models found")

0 models found


In [13]:
def extract_selected_features(model_path, input_UAV_cols):
    with open(model_path, "rb") as f:
        grid = pickle.load(f)

    best_pipe = grid.best_estimator_
    selector = best_pipe.named_steps["feature_selection"]
    mask = selector.get_support()

    assert len(mask) == len(input_UAV_cols)

    selected_features = np.array(input_UAV_cols)[mask]
    
    return selected_features

In [14]:
def get_img_type_from_model_path(model_path):
    fname = os.path.basename(model_path)
    m = re.search(r"ImgType-([A-Za-z]+)", fname)
    if m is None:
        raise ValueError(f"ImgType not found in model name: {fname}")
    return m.group(1)

In [15]:
counter_RGB = Counter()
counter_Full = Counter()
n_models_RGB = 0
n_models_Full = 0

for model_path in model_paths:

    img_type = get_img_type_from_model_path(model_path)

    if img_type == "RGB":
        input_UAV_cols = RGB_input_UAV_cols
        counter = counter_RGB
        n_models_RGB += 1

    elif img_type == "Full":
        input_UAV_cols = RGB_input_UAV_cols + mul_input_UAV_cols
        counter = counter_Full
        n_models_Full += 1

    selected_features = extract_selected_features(model_path, input_UAV_cols)
    counter.update(selected_features)

In [16]:
# RGB
freq_RGB = pd.DataFrame({"feature": RGB_input_UAV_cols})
freq_RGB["count"] = freq_RGB["feature"].map(counter_RGB).fillna(0).astype(int)
freq_RGB["frequency"] = freq_RGB["count"] / n_models_RGB

# Full
freq_Full = pd.DataFrame({"feature": RGB_input_UAV_cols + mul_input_UAV_cols})
freq_Full["count"] = freq_Full["feature"].map(counter_Full).fillna(0).astype(int)
freq_Full["frequency"] = freq_Full["count"] / n_models_Full

In [17]:
feature_order = [
    "CC", "PH", "PV",          # structural traits
    "ExG", "VARI", "GRVI",     # RGB color indices
    "NDVI", "GNDVI", "NDRE"    # multispectral indices
]

In [18]:
def get_feature_prefix(feature_name):
    for key in feature_order:
        if feature_name.startswith(key):
            return key
    return "OTHER"
def feature_sort_key(feature_name):
    prefix = get_feature_prefix(feature_name)
    order_idx = feature_order.index(prefix) if prefix in feature_order else len(feature_order)
    return (order_idx, feature_name)

In [19]:
freq_RGB_plot = freq_RGB 
freq_Full_plot = freq_Full

freq_RGB_plot["sort_key"] = freq_RGB_plot["feature"].apply(feature_sort_key)
freq_RGB_plot = freq_RGB_plot.sort_values("sort_key").drop(columns="sort_key")

freq_Full_plot["sort_key"] = freq_Full_plot["feature"].apply(feature_sort_key)
freq_Full_plot = freq_Full_plot.sort_values("sort_key").drop(columns="sort_key")

In [20]:
def build_input_UAV_cols(
    HarvTime_now,
    img_type,
    latest_flight_date_li,
    flight_date_li,
    mul_flight_date_li,
    RGB_Pheno_names,
    Mul_Pheno_names,
):
    idxs = list(range(1 + HarvTime_now)) 
    input_comb_date_now = [latest_flight_date_li[i] for i in idxs]

    RGB_input_comb_now = [s for s in flight_date_li if s[:8] in input_comb_date_now]
    RGB_input_UAV_cols = [
        f"{p}_{d}"
        for d in RGB_input_comb_now
        for p in RGB_Pheno_names
    ]

    mul_input_comb_now = [s for s in mul_flight_date_li if s[:8] in input_comb_date_now]
    mul_input_UAV_cols = [
        f"{p}_{d}"
        for d in mul_input_comb_now
        for p in Mul_Pheno_names
    ]

    if img_type == "RGB":
        return RGB_input_UAV_cols
    elif img_type == "Full":
        return RGB_input_UAV_cols + mul_input_UAV_cols
    else:
        raise ValueError(f"Unknown img_type: {img_type}")

In [21]:
def collect_model_paths(
    base_dir,
    HarvTime_now,
    CV_modeltime_now,
):
    model_dir = os.path.join(
        base_dir,
        f"CV10_harv{HarvTime_now}",
        f"K10foldCVwithPipe5_{CV_modeltime_now}"
    )

    model_paths_all = glob.glob(os.path.join(model_dir, "*.pickele"))

    model_paths = [
        p for p in model_paths_all
        if ("SampRate-100" in os.path.basename(p))
        and ("SampMetho-random" in os.path.basename(p))
        and ("proto" not in os.path.basename(p))
    ]

    return model_paths

In [22]:
def filter_model_paths_by_img_type(model_paths, img_type):
    return [
        p for p in model_paths
        if f"ImgType-{img_type}" in os.path.basename(p)
    ]

In [23]:
def count_selected_features(
    model_paths,
    input_UAV_cols,
):
    counter = Counter()
    n_models = 0

    for model_path in model_paths:
        selected_features = extract_selected_features(
            model_path,
            input_UAV_cols
        )
        counter.update(selected_features)
        n_models += 1

    freq_df = pd.DataFrame({"feature": input_UAV_cols})
    freq_df["count"] = freq_df["feature"].map(counter).fillna(0).astype(int)
    freq_df["frequency"] = freq_df["count"] / n_models

    return freq_df, n_models

In [24]:
def sort_features_by_paper_order(freq_df):

    def get_prefix(name):
        for k in feature_order:
            if name.startswith(k):
                return k
        return "OTHER"

    def sort_key(name):
        p = get_prefix(name)
        idx = feature_order.index(p) if p in feature_order else len(feature_order)
        return (idx, name)

    df = freq_df.copy()
    df["sort_key"] = df["feature"].apply(sort_key)
    df = df.sort_values("sort_key").drop(columns="sort_key")

    return df

In [25]:
for HarvTime_now in range(1, 5):

    print(f"\n=== HarvTime {HarvTime_now} ===")

    model_paths_all = collect_model_paths(
        base_dir,
        HarvTime_now,
        CV_modeltime_now,
    )

    print(f"{len(model_paths_all)} total models found")

    for img_type in ["RGB", "Full"]:

        model_paths = filter_model_paths_by_img_type(
            model_paths_all,
            img_type
        )

        print(f"  {img_type}: {len(model_paths)} models")

        input_UAV_cols = build_input_UAV_cols(
            HarvTime_now,
            img_type,
            latest_flight_date_li,
            flight_date_li,
            mul_flight_date_li,
            RGB_Pheno_names,
            Mul_Pheno_names,
        )

        freq_df, n_models = count_selected_features(
            model_paths,
            input_UAV_cols,
        )

        freq_df = sort_features_by_paper_order(freq_df)


=== HarvTime 1 ===
0 total models found
  RGB: 0 models
  Full: 0 models

=== HarvTime 2 ===
0 total models found
  RGB: 0 models
  Full: 0 models

=== HarvTime 3 ===
0 total models found
  RGB: 0 models
  Full: 0 models

=== HarvTime 4 ===
0 total models found
  RGB: 0 models
  Full: 0 models


Create a heatmap of feature selection

In [26]:
def split_feature_and_date(feature_name):
    parts = feature_name.split("_")
    feature = parts[0]
    date = parts[2][0:8]
    return feature, date

In [27]:
def make_feature_date_matrix(freq_df):
    rows = []

    for _, r in freq_df.iterrows():
        feat, date = split_feature_and_date(r["feature"])
        rows.append({
            "feature": feat,
            "date": date,
            "frequency": r["frequency"]
        })

    df_long = pd.DataFrame(rows)
    df_mat = (
        df_long
        .pivot_table(
            index="feature",
            columns="date",
            values="frequency",
            aggfunc="mean"
        )
        .fillna(0)
    )

    return df_mat

In [28]:
df_mat = make_feature_date_matrix(freq_df)
df_mat

date
feature


In [29]:
all_dates = sorted(
    {s[:8] for s in flight_date_li[1:5]} |
    {s[:8] for s in mul_flight_date_li[1:5]}
)
all_features = feature_order.copy()

In [30]:
def expand_matrix(df_mat, all_features, all_dates):
    df = df_mat.copy()
    for f in all_features:
        if f not in df.index:
            df.loc[f] = np.nan

    for d in all_dates:
        if d not in df.columns:
            df[d] = np.nan
            
    df = df.loc[all_features, sorted(all_dates)]

    return df

In [31]:
def sort_matrix(df_mat):
    df = df_mat.copy()
    df["__order"] = df.index.map(
        lambda x: feature_order.index(x) if x in feature_order else 999
    )
    df = df.sort_values("__order").drop(columns="__order")

    df = df[df.columns.sort_values()]

    return df

In [32]:
def build_input_mask(input_UAV_cols):
    mask = set()
    for c in input_UAV_cols:
        f, d = split_feature_and_date(c)
        mask.add((f, d))
    return mask

In [33]:
def apply_input_mask(df, input_mask):
    df2 = df.copy()
    for f in df.index:
        for d in df.columns:
            if (f, d) not in input_mask:
                df2.loc[f, d] = np.nan
    return df2

In [34]:
def plot_feature_date_heatmap(df_mat, title, show_cbar=False):

    cmap = sns.color_palette("viridis", as_cmap=True)
    cmap.set_bad(color="lightgray")

    fig = plt.figure(
        figsize=(0.7 * df_mat.shape[1], 0.5 * df_mat.shape[0])
    )

    gs = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[20, 1],   
        wspace=0.05
    )

    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])

    sns.heatmap(
        df_mat,
        cmap=cmap,
        vmin=0,
        vmax=1,
        ax=ax,
        cbar=show_cbar,
        cbar_ax=cax if show_cbar else None,
        linewidths=0.5,
        linecolor="white"
    )

    if not show_cbar:
        cax.set_visible(False)

    ax.set_xlabel("Flight date")
    ax.set_ylabel("Feature")
    ax.set_title(title)

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45,
        ha="right"
    )
    
    fig.subplots_adjust(bottom=0.25)

    return fig, ax

In [35]:
start_time = 1
end_time = 5

for HarvTime_now in range(start_time, end_time):

    print(f"\n=== HarvTime {HarvTime_now} ===")

    model_paths_all = collect_model_paths(
        base_dir,
        HarvTime_now,
        CV_modeltime_now,
    )

    for img_type in ["RGB", "Full"]:

        model_paths = filter_model_paths_by_img_type(
            model_paths_all,
            img_type
        )

        input_UAV_cols = build_input_UAV_cols(
            HarvTime_now,
            img_type,
            latest_flight_date_li,
            flight_date_li,
            mul_flight_date_li,
            RGB_Pheno_names,
            Mul_Pheno_names,
        )

        freq_df, _ = count_selected_features(
            model_paths,
            input_UAV_cols,
        )

        mat = make_feature_date_matrix(freq_df)
        mat = sort_matrix(mat)

        mat = expand_matrix(
            mat,
            all_features=all_features,
            all_dates=all_dates
        )

        input_mask = build_input_mask(input_UAV_cols)
        mat = apply_input_mask(mat, input_mask)
        title_now = f"HarvTime {HarvTime_now} – {img_type}"

        if (HarvTime_now == (end_time-1)) & (img_type == "Full"):
            fig, ax = plot_feature_date_heatmap(
                mat,
                title= title_now,
                show_cbar=True
            )
        else:
            fig, ax = plot_feature_date_heatmap(
                mat,
                title= title_now
            )
        
        
        plt.show()
        fig.savefig(results_dir / f"selected_features_{title_now}.png",
                    bbox_inches="tight", dpi=300)
        plt.close()


=== HarvTime 1 ===


ValueError: cannot set a frame with no defined columns

## Visualization of 10-fold CV 

In [ ]:
def make_res_fig_dir(CV_modeltime_now):
    res_fig_dir = results_dir / f"K10foldCVwithPipe5_{CV_modeltime_now}/"
    return res_fig_dir
res_fig_dir = make_res_fig_dir(CV_modeltime_now)
os.makedirs(res_fig_dir, exist_ok=True)
res_fig_dir

In [ ]:
def summarizing_dataframe(df):
    rep_df_now = df.groupby(["HarvTime", "Flight_date", "img_type", "samp_rate", "samp_method"])[["r","R2","RMSE"]].agg(
        r_mean = ("r","mean"),
        r_std = ("r","std"),            
        R2_mean = ("R2","mean"),
        R2_std = ("R2","std"),
        RMSE_mean = ("RMSE","mean"),
        RMSE_std = ("RMSE","std")
    ).reset_index()
    
    return rep_df_now

In [ ]:
def mod_eva_plot(df, har_now,eva_metho = "R2"):
    fig_num = 1
    fig,ax = plt.subplots(nrows=1,ncols=fig_num, figsize=(fig_num*4,4)) 

    ax1=ax
    ax1.set_xlim(0,110)
    ax1.set_ylim(0,1)
    
    if eva_metho == "RMSE":
        ax1.set_ylim(0,max(df[eva_metho + "_mean"])+100)

    cmap = plt.get_cmap("tab20")
    jit_num = -2
    lines = ["-", ":"]
    marks = ["o", "^"]

    grouped_df = pd.DataFrame(columns= ["HarvTime", "Flight_date", "img_type", "samp_rate", "samp_method"] + [s+("_mean") for s in ("r", "R2", "RMSE")])

    # Plot separately for each img_type
    for type, group_df_now in df.groupby(["img_type", "samp_method"]):
        if type[0] == "RGB":
            col_now = cmap(1)
            mfc_now = "none"
        else:
            col_now = cmap(2)
            mfc_now = col_now 
            
        if type[1] == "random":
            lin_now = lines[0]
            mar_now = marks[0]
            
        else:
            lin_now = lines[1]
            mar_now = marks[1]
        
        ax1.plot(group_df_now['samp_rate'] + jit_num, group_df_now[f'{eva_metho}_mean'],
                 ms = 5, marker = mar_now,markerfacecolor = mfc_now, 
                 lw = 1, linestyle = lin_now,
                 color = col_now, 
                 label=f'{type}')
        
        ax1.errorbar(group_df_now['samp_rate'] + jit_num, group_df_now[f'{eva_metho}_mean'],yerr=group_df_now[f'{eva_metho}_std'], 
                     lw = 0.5, linestyle = lin_now,
                     color = col_now, 
                     capsize=2.5 )

        jit_num +=1

    ax1.set_xlabel(f"sampling_ratio (%)", fontsize=14)
    ax1.set_xlim(15, 105)     
    ax1.set_xticks([25, 50, 75, 100])   
    ax1.set_xticklabels(['25', '50', '75', '100'], fontsize=12)    
    ax1.set_ylabel(f'{eva_metho}', fontsize=14) 
    ax.tick_params(direction='in', labelsize=12) 
  
    lines, labels = ax1.get_legend_handles_labels()
    ax1.set_title(f"{samp_DAS_dic[str(har_now)]} DAS in 2023")

    fig_legend = plt.figure(figsize=(4, 4))
    fig_legend.legend(
        handles=lines,
        labels=labels,
        loc='center',
        frameon=False,
        ncol=1,
        fontsize=12
    )
    fig_legend.tight_layout()
    fig_legend.savefig(results_dir / f"2023_legend_{har_now}.png", dpi=600)
    plt.close(fig_legend)    
    
    if eva_metho == "RMSE":
        ax1.set_ylabel(f'{eva_metho} (g m$^{{-2}}$)', fontsize=14)    

    elif eva_metho == "R2":
        ax1.set_ylabel(r'$R^2$', fontsize=14)
    
    return fig

In [ ]:
def get_resu_df(def_dir,CV_modeltime_now, har_num): 
    if har_num == "all":
        for har_num in range(1,5):
            if har_num == 1:
                resu_df = pd.read_csv(def_dir/ f"CV10_harv{har_num}"/ f"K10foldCVwithPipe5_{CV_modeltime_now}"/ f"resu_df_{har_num}.csv", index_col=0)
                resu_df["har_num"] = har_num
                
            else:
                resu_df_now = pd.read_csv(def_dir / f"CV10_harv{har_num}"/ f"K10foldCVwithPipe5_{CV_modeltime_now}"/ f"resu_df_{har_num}.csv", index_col=0) 
                resu_df_now["har_num"] = har_num
                resu_df = pd.concat([resu_df,resu_df_now])
                
                
    else:
        resu_df = pd.read_csv(def_dir / f"CV10_harv{har_num}"/ f"K10foldCVwithPipe5_{CV_modeltime_now}"/ f"resu_df_{har_num}.csv", index_col=0)
    
    return resu_df
    

In [ ]:
CV_res_df = get_resu_df(base_dir, 
                        CV_modeltime_now, 
                        "all")
summa_CV_res_df = summarizing_dataframe(CV_res_df)
summa_CV_res_df

In [ ]:
conf_summa_CV_res_df = summa_CV_res_df[(summa_CV_res_df["img_type"] == "Full") & (summa_CV_res_df["samp_rate"] == 100) & (summa_CV_res_df["samp_method"] == "random")]
display(conf_summa_CV_res_df)
conf_summa_CV_res_df.to_csv(res_fig_dir / f"{year_now}_summa_CV_res_df.csv")

In [ ]:
def make_mod_evaplot(def_dir,CV_modeltime_now, har_num, eva_metho = "R2"):    
    resu_df_now = get_resu_df(def_dir,CV_modeltime_now, har_num)
    
    metrics = ["R2", "r", "RMSE"]
    
    if eva_metho == "all":
        eva_metho_li = metrics
        
    else:
        eva_metho_li = [eva_metho]
        
    for i in range(len(eva_metho_li)):
        eva_metho_now = eva_metho_li[i]
        
        res_fig_dir = f"{results_dir}/K10foldCVwithPipe5_{CV_modeltime_now}"
        os.makedirs(res_fig_dir, exist_ok=True)
        fig = mod_eva_plot(summarizing_dataframe(resu_df_now),har_num, eva_metho_now)
        fig.savefig(f"{res_fig_dir}/mod_evaplot_{har_num}_{eva_metho_now}.png",
                    dpi=600,
                    bbox_inches="tight") 
        plt.show(fig)
        plt.close(fig)

In [ ]:
for har_num in range(1,5):
    print(har_num)
    make_mod_evaplot(base_dir,
                     CV_modeltime_now, har_num,
                     eva_metho = "all") 

## Visualization of Block-wise Cross-Validation

In [ ]:
def summarizing_dataframe_bloCV(df):
    rep_df_now = df.groupby(["HarvTime", "Flight_date", "img_type", "samp_rate", "samp_method","prediction_modes"])[["r","R2","RMSE"]].agg(
        r_mean = ("r","mean"),
        r_std = ("r","std"),            
        R2_mean = ("R2","mean"),
        R2_std = ("R2","std"),
        RMSE_mean = ("RMSE","mean"),
        RMSE_std = ("RMSE","std")
    ).reset_index()
    
    return rep_df_now

In [ ]:
class ana_bloCV():
    def __init__(self, harvtime, def_dir, blo_CV_modeltime):
        self.harvtime = harvtime
        self.def_dir = def_dir 
        self.blo_CV_modeltime = blo_CV_modeltime 
        
        self.df = pd.read_csv(self.def_dir / 
                              f"CV10_harv{self.harvtime}"/ 
                              f"K10foldBlockCVwithPipe4_{self.blo_CV_modeltime}"/
                              f"resu_df_{self.harvtime}.csv", index_col=0)
        self.rep_df = None
        self.fig_mod_eva_plot = None
        self.fig_boxplot = None

    def make_rep_df(self):
        self.rep_df = self.df.groupby(["HarvTime", "Flight_date", "img_type", "samp_rate", "samp_method","prediction_modes"])[["r","R2","RMSE"]].agg(
            r_mean = ("r","mean"),
            r_std = ("r","std"),            
            R2_mean = ("R2","mean"),
            R2_std = ("R2","std"),
            RMSE_mean = ("RMSE","mean"),
            RMSE_std = ("RMSE","std")
        ).reset_index()
        
    
    def make_mod_eva_plot(self, do_show = False):
        self.make_rep_df()

        eva_metho_li = ["r","R2","RMSE"]
        fig_num = len(eva_metho_li)
        fig,axes = plt.subplots(nrows=1,ncols=fig_num, figsize=(fig_num*4,4)) 
        
        for i in range(len(eva_metho_li)):
            eva_metho_now = eva_metho_li[i]
            ax = axes[i]
            ax.set_xlim(0,110)
            ax.set_ylim(0,1)

            if eva_metho_now == "RMSE":
                ax.set_ylim(0,max(self.rep_df[eva_metho_now + "_mean"])+100)

            cmap = plt.get_cmap("tab20")
            jit_num = -4
            lines = ["-", ":"]
            marks = ["o", "^"]

            for type, group_df_now in self.rep_df.groupby(["prediction_modes"]): 
                if type[0] == "group_within": 
                    col_now = cmap(1)
                    mfc_now = "none"
                elif type[0] == "group_between":
                    col_now = cmap(2)
                    mfc_now = col_now 

                else:
                    col_now = cmap(3)
                    mfc_now = col_now 
                    
                lin_now = lines[0]
                mar_now = marks[0]

                ax.plot(group_df_now['samp_rate'] + jit_num, group_df_now[f'{eva_metho_now }_mean'],
                        ms = 5, marker = mar_now,markerfacecolor = mfc_now, 
                         lw = 1, linestyle = "none", 
                         color = col_now, 
                         label=f'{eva_metho_now } - {type}')
                ax.errorbar(group_df_now['samp_rate'] + jit_num, group_df_now[f'{eva_metho_now }_mean'],yerr=group_df_now[f'{eva_metho_now }_std'], 
                             lw = 0.5,linestyle = "none", 
                             color = col_now, 
                             capsize=2.5 )
                jit_num +=2

            ax.set_xlabel("sampling_ratio")
            ax.set_ylabel(f'{eva_metho_now}')
            
            lines, labels = ax.get_legend_handles_labels()
            ax.legend(lines, labels , loc='lower right') 
            ax.set_title(f"{self.harvtime}")

        if do_show == True:
            plt.show()
        self.mod_eva_plot = fig
        plt.close()
            
    def make_boxplot(self, do_show = False, do_save = False):
        eva_metho_li = ["R2"]  
        fig_num = len(eva_metho_li)
        fig,axes = plt.subplots(nrows=fig_num,ncols=1, figsize=(8, fig_num*4) )
        if fig_num == 1:
            axes = [axes]  
        colors = ["lightgray", "gray", "darkgray", "black"] 
        
        for row in range(len(eva_metho_li)):
            eva_metho_now = eva_metho_li[row]
            
            ax = axes[row]

            ax.set_ylim(-1,1)
            if eva_metho_now == "RMSE": ax.set_ylim(0,700)

            df_now = self.df
            df_now["pred_mode"] = df_now["prediction_modes"]
            df_now.loc[df_now["pred_mode"] == "overall", "pred_mode"] = "_overall" 

            grouped_data = [group[eva_metho_now] for _, group in self.df.groupby(["samp_rate", "pred_mode","img_type", 'samp_method'])]

            labels = [
                f"{samp_rate}-{pred_mode.split('_')[1]}-{img_type[0]}-{samp_method[0]}" 
                for samp_rate,pred_mode, img_type, samp_method,  
                in self.df.groupby(["samp_rate","pred_mode","img_type", 'samp_method']).groups.keys()]

            ax.boxplot(grouped_data, labels=labels, patch_artist=True,
                      medianprops=dict(color='black', linewidth=1)  
                      )
            
            if eva_metho_now == "R2":
                ax.set_ylabel( r"$R^2$", fontsize=16)
            else:
                ax.set_ylabel(label, fontsize=16)
        
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.tick_params(direction='in') 
            ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=14)
            stra_label_set =  [label for label, group in self.df.groupby(["img_type", 'samp_method'])]
            
            for samp_rate_i in range(len(self.df["samp_rate"].unique())):
                ax.axvspan(samp_rate_i*(len(labels)//len(self.df["samp_rate"].unique()))+0.5,
                           (samp_rate_i+1)*(len(labels)//len(self.df["samp_rate"].unique()))+0.5, 
                           facecolor=colors[samp_rate_i], alpha=0.3)
                
                for stra_label_i in range(len(self.df["pred_mode"].unique())):
                    ax.axvline(samp_rate_i*(len(labels)//len(self.df["samp_rate"].unique())) + 0.5 + (len(stra_label_set)*stra_label_i),
                                color='black', linestyle='--')
        fig.suptitle(f"{samp_DAS_dic[str(har_now)]} DAS in 2023",fontsize=16, y = 0.92)           
        plt.tight_layout(rect=[0, 0, 1, 0.99])
        
        if do_show == True:
            plt.show()
        self.fig_boxplot = fig
        
        if do_save == True:
            fig.savefig(f"{results_dir}boxplot_{str(self.harvtime)}_{self.blo_CV_modeltime}")
            
        plt.close()           

In [ ]:
for har_now in range(1,5):
    inst_har = ana_bloCV(harvtime=har_now,
                         def_dir = base_dir,
                         blo_CV_modeltime = blo_CV_modeltime_now)
    inst_har.make_boxplot(do_show=True, do_save = True) 